In [1]:
import pandas as pd

In [2]:
path=r"C:\projects\nz-credit-risk-lab\data\accepted_2007_to_2018Q4.csv"

In [3]:
header=pd.read_csv(path, nrows=0)

In [4]:
all_cols=list(header.columns)

In [5]:
len(all_cols)

151

In [6]:
print(all_cols)

['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'acc_now_delinq',

In [7]:
cols=["id", "loan_amnt", "term", "int_rate", "grade", "sub_grade","emp_length", "home_ownership","annual_inc","purpose","dti","issue_d", "loan_status"]

In [8]:
missing=[c for c in cols if c not in all_cols]

In [9]:
df = pd.read_csv(path, usecols=cols, low_memory=False)
df.shape

(2260701, 13)

In [10]:
print(len(all_cols))
print(missing)
print(df.shape)

151
[]
(2260701, 13)


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 13 columns):
 #   Column          Dtype  
---  ------          -----  
 0   id              object 
 1   loan_amnt       float64
 2   term            object 
 3   int_rate        float64
 4   grade           object 
 5   sub_grade       object 
 6   emp_length      object 
 7   home_ownership  object 
 8   annual_inc      float64
 9   issue_d         object 
 10  loan_status     object 
 11  purpose         object 
 12  dti             float64
dtypes: float64(4), object(9)
memory usage: 224.2+ MB


In [12]:
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 13 columns):
 #   Column          Dtype  
---  ------          -----  
 0   id              object 
 1   loan_amnt       float64
 2   term            object 
 3   int_rate        float64
 4   grade           object 
 5   sub_grade       object 
 6   emp_length      object 
 7   home_ownership  object 
 8   annual_inc      float64
 9   issue_d         object 
 10  loan_status     object 
 11  purpose         object 
 12  dti             float64
dtypes: float64(4), object(9)
memory usage: 1.1 GB


In [15]:
df["term"].value_counts()

term
36 months    1609754
60 months     650914
Name: count, dtype: int64

In [18]:
df["issue_d"]=pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce")
print(df["issue_d"].min())
print(df["issue_d"].max())

2007-06-01 00:00:00
2018-12-01 00:00:00


In [23]:
mask=(df["term"].str.strip()=="36 months") & (df["issue_d"].dt.year.between(2012,2015))
df36=df[mask].copy()
df36.shape

(589635, 13)

In [24]:
df36["loan_status"].value_counts()

loan_status
Fully Paid            506760
Charged Off            82728
Current                   72
Late (31-120 days)        61
In Grace Period            9
Late (16-30 days)          5
Name: count, dtype: int64

In [25]:
df36=df36[df36["loan_status"].isin(["Fully Paid", "Charged Off", "Default"])].copy()
df36.shape

(589488, 13)

In [26]:
df36["bad"] = df36["loan_status"].isin(["Charged Off", "Default"]).astype(int)
print(df36["bad"].value_counts())
print(df36["bad"].mean())

bad
0    506760
1     82728
Name: count, dtype: int64
0.1403387346307304


In [27]:
print(df36.shape)
print(df36["bad"].value_counts())
print(df36["bad"].mean())

(589488, 14)
bad
0    506760
1     82728
Name: count, dtype: int64
0.1403387346307304


In [28]:
df36.isnull().sum()

id                    0
loan_amnt             0
term                  0
int_rate              0
grade                 0
sub_grade             0
emp_length        35330
home_ownership        0
annual_inc            0
issue_d               0
loan_status           0
purpose               0
dti                   2
bad                   0
dtype: int64

In [29]:
df36.groupby("grade")["bad"].mean()

grade
A    0.054480
B    0.112213
C    0.180322
D    0.235655
E    0.289165
F    0.331441
G    0.401728
Name: bad, dtype: float64

In [30]:
df36.groupby(df36["emp_length"].isnull())["bad"].mean()

emp_length
False    0.136203
True     0.205208
Name: bad, dtype: float64

In [31]:
df36["emp_length"].value_counts(dropna=False)

emp_length
10+ years    184504
2 years       53446
< 1 year      47795
3 years       47348
1 year        39094
5 years       37131
NaN           35330
4 years       35030
7 years       29553
8 years       29174
6 years       28616
9 years       22467
Name: count, dtype: int64

In [32]:
emp_map = {"< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3,
           "4 years": 4, "5 years": 5, "6 years": 6, "7 years": 7,
           "8 years": 8, "9 years": 9, "10+ years": 10}

In [33]:
df36["emp_years"] = df36["emp_length"].map(emp_map)

In [34]:
df36["emp_missing"] = df36["emp_length"].isnull().astype(int)

In [35]:
print(df36["emp_years"].value_counts(dropna=False).sort_index())
print(df36["emp_missing"].sum())

emp_years
0.0      47795
1.0      39094
2.0      53446
3.0      47348
4.0      35030
5.0      37131
6.0      28616
7.0      29553
8.0      29174
9.0      22467
10.0    184504
NaN      35330
Name: count, dtype: int64
35330


In [36]:
df36 = df36.dropna(subset=["dti"]).copy()
df36.shape

(589486, 16)